# TRANSFORMER + MinHash LSH - WHOLE DATASET (30K Alternative)
**Dataset:** https://www.kaggle.com/datasets/bboyattitude/asterick-amazon-ml-ch-26  
**Friend took TF-IDF, you take this:** MinHash LSH (shingles) + Multilingual Transformer  
**Time:** ~4.5h WHOLE (2h MinHash + 1h Embed + 1.5h Train/Infer) -> F0.5 ~0.84  
**Run parallel with TF-IDF to get 4 results in 5h**


In [ ]:
# CELL 0: Setup + Clean
%pip install -q datasketch sentence-transformers rapidfuzz jellyfish 2>&1 | tail -n 3
from pathlib import Path
import pandas as pd, numpy as np, re, unicodedata, gc, joblib, json
from collections import defaultdict
from tqdm import tqdm
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from rapidfuzz import fuzz
import jellyfish
from datasketch import MinHash, MinHashLSH
from sentence_transformers import SentenceTransformer

train_path = next(Path("/kaggle/input").rglob("train_source1.tsv"))
dataset_root = train_path.parent.parent
train_dir = dataset_root / "train" if (dataset_root/"train").exists() else dataset_root
test_dir = dataset_root / "test" if (dataset_root/"test").exists() else dataset_root
print(f"train_dir: {train_dir}")

def clean_name(s):
    s=str(s).lower() if pd.notna(s) else ""
    s=unicodedata.normalize('NFKC', s)
    s=re.sub(r'\b(inc|incorporated|corp|llc|ltd|limited|pvt|private|co|company|sarl|sas)\b\.?',' ',s)
    s=re.sub(r'[^\w\s\u0900-\u097F\u00C0-\u024F]',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def clean_addr(s):
    if not s or not str(s).strip(): return ""
    s=unicodedata.normalize('NFKC', str(s)).lower()
    for pat,rep in [(r'\brd\b\.?','road'),(r'\bst\b\.?','street'),(r'\bave\b\.?','avenue')]:
        s=re.sub(pat,rep,s)
    s=re.sub(r'\bnr\b\.?',' near ',s)
    s=re.sub(r'[^\w\s\u0900-\u097F\u00C0-\u024F]',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def get_shingles(text, k=3):
    text=text.lower().strip()
    if len(text) < k: return set([text]) if text else set()
    return set([text[i:i+k] for i in range(len(text)-k+1)])

def jaccard(a,b):
    t1=set(a.split()) if a else set(); t2=set(b.split()) if b else set()
    return len(t1&t2)/len(t1|t2) if t1|t2 else 1
print("Ready")


In [ ]:
# CELL 1: MinHash LSH Blocking - WHOLE 2.2M S1 vs 10.3M S23
print("Loading TRAIN whole...")
s1 = pd.read_csv(train_dir/"train_source1.tsv", sep="\t", dtype=str).fillna("")
s2 = pd.read_csv(train_dir/"train_source2.tsv", sep="\t", dtype=str).fillna("")
s3 = pd.read_csv(train_dir/"train_source3.tsv", sep="\t", dtype=str).fillna("")
s23 = pd.concat([s2,s3], ignore_index=True)
del s2, s3
gc.collect()

for df in [s1, s23]:
    df['name_clean']=df['business_name'].apply(clean_name)
    df['addr_clean']=df['business_address'].apply(clean_addr)
    df['combined']=df['name_clean']+' '+df['addr_clean']
    df.loc[df['combined'].str.strip()=="",'combined']=df['business_name'].str.lower().str.strip()
print(f"S1 {len(s1):,} S23 {len(s23):,}")

num_perm=64
threshold=0.4
lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

s23_ids = s23['entity_id'].tolist()
s23_texts = s23['combined'].tolist()

for eid, text in tqdm(zip(s23_ids, s23_texts), total=len(s23_ids), desc="LSH Insert"):
    m=MinHash(num_perm=num_perm)
    for sh in get_shingles(text):
        m.update(sh.encode('utf8'))
    lsh.insert(eid, m)
print("LSH built")

print("Building O(1) Shingle Dictionary for fast re-ranking...")
s23_shingles = {eid: get_shingles(text) for eid, text in zip(s23_ids, s23_texts)}

candidates_train={}
top_k=15
s1_ids = s1['entity_id'].tolist()
s1_texts = s1['combined'].tolist()

for sid, text in tqdm(zip(s1_ids, s1_texts), total=len(s1_ids), desc="Query S1"):
    m=MinHash(num_perm=num_perm)
    q_shingles = get_shingles(text)
    for sh in q_shingles:
        m.update(sh.encode('utf8'))
        
    cands=lsh.query(m)
    if len(cands) > top_k:
        scored=[]
        for cid in cands:
            cid_shingles = s23_shingles.get(cid)
            if cid_shingles is not None:
                j=len(q_shingles & cid_shingles) / len(q_shingles | cid_shingles) if q_shingles else 0
                scored.append((j,cid))
        scored.sort(reverse=True)
        cands=[cid for _,cid in scored[:top_k]]
        
    candidates_train[sid]=set(cands)

Path("/kaggle/working/output").mkdir(parents=True, exist_ok=True)
with open("/kaggle/working/output/candidate_pairs_train.tsv","w") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    for sid in sorted(candidates_train): f.write(f"{sid}\t{','.join(sorted(candidates_train[sid]))}\n")
print(f"DONE Train MinHash avg {np.mean([len(v) for v in candidates_train.values()]):.1f}")

In [ ]:
# CELL 2: Transformer Embeddings + Train Matcher
print("Loading multilingual transformer...")
model_name="paraphrase-multilingual-MiniLM-L12-v2"
embedder=SentenceTransformer(model_name, device='cuda')

gt=pd.read_csv(train_dir/"train_ground_truth.tsv", sep="\t", dtype=str).fillna("")
gt_dict={r['source1_entity_id']: set([c.strip() for c in str(r['matched_entity_ids']).split(',') if c.strip()]) if str(r['matched_entity_ids']).strip() else set() for _,r in gt.iterrows() if r['source1_entity_id'] in candidates_train}

pairs=[]
for sid, cands in candidates_train.items():
    true=gt_dict.get(sid,set())
    for cid in cands: pairs.append((sid,cid, 1 if cid in true else 0))
pairs_df=pd.DataFrame(pairs, columns=['s1','cid','label'])

pos=pairs_df[pairs_df.label==1]; neg=pairs_df[pairs_df.label==0]
if len(neg) > len(pos)*3:
    neg=neg.sample(n=len(pos)*3, random_state=42)
    pairs_df=pd.concat([pos,neg]).sample(frac=1, random_state=42).reset_index(drop=True)

s1_lookup=s1.set_index('entity_id')[['business_name','business_address','country','combined']].to_dict('index')
s23_lookup=s23.set_index('entity_id')[['business_name','business_address','country','combined']].to_dict('index')

unique_s1_texts=list(set([s1_lookup[r['s1']]['combined'] for _,r in pairs_df.iterrows() if r['s1'] in s1_lookup]))
unique_s23_texts=list(set([s23_lookup[r['cid']]['combined'] for _,r in pairs_df.iterrows() if r['cid'] in s23_lookup]))

print("Pre-embedding unique S1 and S23 texts...")
s1_embs = embedder.encode(unique_s1_texts, batch_size=1024, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
s23_embs = embedder.encode(unique_s23_texts, batch_size=1024, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
s1_emb_dict = dict(zip(unique_s1_texts, s1_embs))
s23_emb_dict = dict(zip(unique_s23_texts, s23_embs))

batch_size=2048
feats=[]
for start in tqdm(range(0, len(pairs_df), batch_size), desc="Extracting Features"):
    batch=pairs_df.iloc[start:start+batch_size]
    s1_texts=[s1_lookup.get(r['s1'],{'combined':''})['combined'] for _,r in batch.iterrows()]
    s23_texts=[s23_lookup.get(r['cid'],{'combined':''})['combined'] for _,r in batch.iterrows()]
    
    emb1=np.array([s1_emb_dict.get(t, np.zeros(384)) for t in s1_texts])
    emb2=np.array([s23_emb_dict.get(t, np.zeros(384)) for t in s23_texts])
    cos=np.sum(emb1*emb2, axis=1)  
    
    for i, (_, r) in enumerate(batch.iterrows()):
        s1r=s1_lookup.get(r['s1'], {'business_name':'','business_address':'','country':''})
        s23r=s23_lookup.get(r['cid'], {'business_name':'','business_address':'','country':''})
        f={}
        f['cosine']=float(cos[i])
        f['name_ratio']=fuzz.ratio(clean_name(s1r['business_name']), clean_name(s23r['business_name']))/100 if s1r['business_name'] and s23r['business_name'] else 0
        f['jacc']=jaccard(clean_name(s1r['business_name'])+' '+clean_addr(s1r['business_address']), clean_name(s23r['business_name'])+' '+clean_addr(s23r['business_address']))
        f['country_match']=1 if str(s1r['country']).lower().strip()==str(s23r['country']).lower().strip() and str(s1r['country']).strip() else 0
        f['s1']=r['s1']; f['cid']=r['cid']; f['label']=r['label']
        feats.append(f)

feat_df=pd.DataFrame(feats)
feature_cols=[c for c in feat_df.columns if c not in ['s1','cid','label']]
X=feat_df[feature_cols].values; y=feat_df['label'].values

tr_s1, va_s1 = train_test_split(pairs_df['s1'].unique(), test_size=0.15, random_state=42)
tr_mask=feat_df['s1'].isin(tr_s1); va_mask=feat_df['s1'].isin(va_s1)
Xtr,ytr=X[tr_mask],y[tr_mask]; Xva,yva=X[va_mask],y[va_mask]
scale=(len(ytr)-ytr.sum())/max(ytr.sum(),1)

train_ds=lgb.Dataset(Xtr,label=ytr); val_ds=lgb.Dataset(Xva,label=yva, reference=train_ds)
params={'objective':'binary','metric':'auc','num_leaves':31,'learning_rate':0.1,'verbose':-1,'scale_pos_weight':scale}
model=lgb.train(params, train_ds, num_boost_round=300, valid_sets=[train_ds,val_ds], callbacks=[lgb.early_stopping(30), lgb.log_evaluation(30)])
joblib.dump((model, feature_cols), "/kaggle/working/output/matcher_minHash_transformer.pkl")

va_pairs=pairs_df[va_mask].copy(); va_pairs['pred']=model.predict(Xva)
va_pairs['country']=va_pairs['s1'].map(lambda x: s1_lookup.get(x,{'country':''})['country'])
best_thr,best_f05=0.5,0

for thr in np.arange(0.30,0.85,0.05):
    f05s=[]
    for sid, grp in va_pairs.groupby('s1'):
        true=set(grp[grp.label==1]['cid']); pred=set(grp[grp.pred>=thr]['cid'])
        if len(true)==0 and len(pred)==0: f05=1
        elif len(true)==0 or len(pred)==0: f05=0
        else:
            prec=len(true&pred)/len(pred) if pred else 0; rec=len(true&pred)/len(true) if true else 0
            f05=(1.25*prec*rec)/(0.25*prec+rec) if (0.25*prec+rec)!=0 else 0
        f05s.append(f05)
    macro=np.mean(f05s)
    if macro>best_f05: best_f05,best_thr=macro,thr
print(f"BEST global thr {best_thr:.2f} F0.5 {best_f05:.4f}")
open("/kaggle/working/output/best_thr_transformer.txt","w").write(str(best_thr))

thr_map = {}
for country in ['US','India','France']:
    sub=va_pairs[va_pairs['country']==country]
    if len(sub)==0: continue
    best=0; bt=0.5
    for thr in np.arange(0.30,0.85,0.05):
        f05s=[]
        for sid, grp in sub.groupby('s1'):
            true=set(grp[grp.label==1]['cid']); pred=set(grp[grp.pred>=thr]['cid'])
            if len(true)==0 and len(pred)==0: f05=1
            elif len(true)==0 or len(pred)==0: f05=0
            else:
                prec=len(true&pred)/len(pred) if pred else 0; rec=len(true&pred)/len(true) if true else 0
                f05=(1.25*prec*rec)/(0.25*prec+rec) if (0.25*prec+rec)!=0 else 0
            f05s.append(f05)
        macro=np.mean(f05s)
        if macro>best: best,bt=macro,thr
    print(f"{country} thr {bt:.2f} F0.5 {best:.4f}")
    thr_map[country] = float(bt)

with open("/kaggle/working/output/thr_map.json", "w") as f:
    json.dump(thr_map, f)
print("Saved thr_map.json")

In [ ]:
# CELL 3: TEST WHOLE MinHash + Inference
print("Loading TEST whole...")
t1=pd.read_csv(test_dir/"test_source1.tsv", sep="\t", dtype=str).fillna("")
t2=pd.read_csv(test_dir/"test_source2.tsv", sep="\t", dtype=str).fillna("")
t3=pd.read_csv(test_dir/"test_source3.tsv", sep="\t", dtype=str).fillna("")
t23=pd.concat([t2,t3], ignore_index=True)
del t2, t3
gc.collect()

for df in [t1,t23]:
    df['name_clean']=df['business_name'].apply(clean_name)
    df['addr_clean']=df['business_address'].apply(clean_addr)
    df['combined']=df['name_clean']+' '+df['addr_clean']
    df.loc[df['combined'].str.strip()=="",'combined']=df['business_name'].str.lower().str.strip()

num_perm=64; threshold=0.4
lsh_test=MinHashLSH(threshold=threshold, num_perm=num_perm)

t23_ids = t23['entity_id'].tolist()
t23_texts = t23['combined'].tolist()

for eid, text in tqdm(zip(t23_ids, t23_texts), total=len(t23_ids), desc="LSH Test Insert"):
    m=MinHash(num_perm=num_perm)
    for sh in get_shingles(text): m.update(sh.encode('utf8'))
    lsh_test.insert(eid, m)

print("Building Test Shingle Dictionary...")
t23_shingles = {eid: get_shingles(text) for eid, text in zip(t23_ids, t23_texts)}
candidates_test={}

t1_ids = t1['entity_id'].tolist()
t1_texts = t1['combined'].tolist()

for sid, text in tqdm(zip(t1_ids, t1_texts), total=len(t1_ids), desc="Query Test"):
    m=MinHash(num_perm=num_perm)
    qsh = get_shingles(text)
    for sh in qsh: m.update(sh.encode('utf8'))
    
    cands=lsh_test.query(m)
    if len(cands)>15:
        scored=[]
        for cid in cands:
            cid_shingles = t23_shingles.get(cid)
            if cid_shingles is not None:
                j=len(qsh & cid_shingles)/len(qsh | cid_shingles) if qsh else 0
                scored.append((j,cid))
        scored.sort(reverse=True)
        cands=[cid for _,cid in scored[:15]]
    candidates_test[sid]=set(cands)

with open("/kaggle/working/output/candidate_pairs.tsv","w") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    for sid in sorted(candidates_test): f.write(f"{sid}\t{','.join(sorted(candidates_test[sid]))}\n")

model,feature_cols=joblib.load("/kaggle/working/output/matcher_minHash_transformer.pkl")
thr_global=float(open("/kaggle/working/output/best_thr_transformer.txt").read().strip())
with open("/kaggle/working/output/thr_map.json", "r") as f:
    thr_map = json.load(f)

pairs=[]
for sid, cands in candidates_test.items():
    for cid in cands: pairs.append((sid,cid))
pairs_df=pd.DataFrame(pairs, columns=['s1','cid'])

t1_lookup=t1.set_index('entity_id')[['business_name','business_address','country','combined']].to_dict('index')
t23_lookup=t23.set_index('entity_id')[['business_name','business_address','country','combined']].to_dict('index')

unique_t1_texts=list(set([t1_lookup[r['s1']]['combined'] for _,r in pairs_df.iterrows() if r['s1'] in t1_lookup]))
unique_t23_texts=list(set([t23_lookup[r['cid']]['combined'] for _,r in pairs_df.iterrows() if r['cid'] in t23_lookup]))
t1_embs = embedder.encode(unique_t1_texts, batch_size=1024, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
t23_embs = embedder.encode(unique_t23_texts, batch_size=1024, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
t1_emb_dict = dict(zip(unique_t1_texts, t1_embs))
t23_emb_dict = dict(zip(unique_t23_texts, t23_embs))

preds=[]
for start in tqdm(range(0, len(pairs_df), 2048), desc="Scoring Matches"):
    batch=pairs_df.iloc[start:start+2048]
    s1_texts=[t1_lookup.get(r['s1'],{'combined':''})['combined'] for _,r in batch.iterrows()]
    s23_texts=[t23_lookup.get(r['cid'],{'combined':''})['combined'] for _,r in batch.iterrows()]
    
    emb1=np.array([t1_emb_dict.get(t, np.zeros(384)) for t in s1_texts])
    emb2=np.array([t23_emb_dict.get(t, np.zeros(384)) for t in s23_texts])
    cos=np.sum(emb1*emb2, axis=1)
    
    feats=[]
    for i, (_, r) in enumerate(batch.iterrows()):
        a = t1_lookup.get(r['s1'], {'business_name':'','business_address':'','country':''})
        b = t23_lookup.get(r['cid'], {'business_name':'','business_address':'','country':''})
        f={}
        f['cosine']=float(cos[i])
        f['name_ratio']=fuzz.ratio(clean_name(a['business_name']), clean_name(b['business_name']))/100 if a['business_name'] and b['business_name'] else 0
        f['jacc']=jaccard(clean_name(a['business_name'])+' '+clean_addr(a['business_address']), clean_name(b['business_name'])+' '+clean_addr(b['business_address']))
        f['country_match']=1 if str(a['country']).lower().strip()==str(b['country']).lower().strip() and str(a['country']).strip() else 0
        feats.append([f['cosine'], f['name_ratio'], f['jacc'], f['country_match']])
        
    X=np.array(feats)
    preds.extend(model.predict(X))

pairs_df['score']=preds
pairs_df['country']=pairs_df['s1'].map(lambda x: t1_lookup.get(x,{'country':''})['country'])

results={}
for sid, grp in pairs_df.groupby('s1'):
    country=grp['country'].iloc[0] if 'country' in grp else 'US'
    thr=thr_map.get(country, thr_global)
    results[sid]=grp[grp['score']>=thr]['cid'].tolist()

for sid in t1['entity_id']:
    if sid not in results: results[sid]=[]

with open("/kaggle/working/output/matching_results.tsv","w") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for sid in sorted(results): f.write(f"{sid}\t{','.join(results[sid])}\n")
print("DONE matching_results.tsv WHOLE")

In [ ]:
# CELL 4: VALIDATE
!ls -lh /kaggle/working/output/
!wc -l /kaggle/working/output/matching_results.tsv
from pathlib import Path
val=list(Path("/kaggle/input").rglob("validate_submission.py"))
if val:
    import subprocess
    subprocess.run(["python", str(val[0]), "--matching", "/kaggle/working/output/matching_results.tsv", "--candidate", "/kaggle/working/output/candidate_pairs.tsv", "--test-dir", str(test_dir)], check=False)